# Quantum teleportation using Bell measurement

This file outlines the process of Alice's initial arbitrary vector called |\psi⟩ being teleported to Bob. An EPR source prepares a Bell state: $\frac{1}{\sqrt(2)}(|00⟩+|11\rangle)$. One qubit is sent to Alice and the other is sent to Bob. Thus, they share an entangled pair of qubits before we can begin.

Now, the goal is to transfer Alice's quantum state's information to Bob but she cannot create a copy of this state due to the no-cloning theorem. Let's consider that she initially has the zeroth qubit and uses a Projector $P$ for each of the Pauli gate matrices: $I, X, Y, Z$. So Alice now has two qubits: one from the zeroth qubit of the EPR pair entangled and the other from Alice's initial $|\psi⟩$ that needs to be teleported to Bob.

We will then use a Bell state measurement for both these qubits by taking the tensor product of her initial arbitrary $|\psi⟩$ with the prepared EPR pair $\frac{1}{\sqrt(2)}(|00⟩+|11\rangle)$.

Upon Bell state measurement, the tensor product collapses to one of the four possible Bell states and all Bob has to do is apply one of the four Pauli gates to deduce Alice's $|\psi⟩$.

In [ ]:
import numpy as np
import scipy as sp

In [ ]:
# Alice's arbitrary initial qubit

arb_num = np.random.rand()

alpha = np.sqrt(arb_num)
beta = np.sqrt(1 - arb_num) # the state must be normalized to 1 so alpha^2 + beta^2 =1

alice_psi = np.array(
    [[alpha],
    [beta]],
    )
print("Alice's initial qubit:")
print(alice_psi)

norm = np.linalg.norm(alice_psi)
print(f"Norm: {norm}")

Alice's initial qubit:
[[0.88659697]
 [0.46254276]]
Norm: 1.0


### Bob's $4 \times 1$ EPR Pair

The EPR source emits the Bell state:

$$|\Phi^+\rangle = \frac{1}{\sqrt{2}} (|00\rangle + |11\rangle)$$

In the 2-qubit computational basis $\{|00\rangle, |01\rangle, |10\rangle, |11\rangle\}$, this is a $4 \times 1$ vector.

**Qubit ordering:**
* **First index:** Qubit 1 (Alice's)
* **Second index:** Qubit 2 (Bob's)

**Basis Vectors:**
* $|00\rangle = [1, 0, 0, 0]^T$
* $|01\rangle = [0, 1, 0, 0]^T$
* $|10\rangle = [0, 0, 1, 0]^T$
* $|11\rangle = [0, 0, 0, 1]^T$

In [ ]:
ket00 = np.array([[1], [0], [0], [0]], dtype = np.complex128)
ket01 = np.array([[0], [1], [0], [0]], dtype = np.complex128)
ket10 = np.array([[0], [0], [1], [0]], dtype = np.complex128)
ket11 = np.array([[0], [0], [0], [1]], dtype = np.complex128)

epr_pair = 1/np.sqrt(2) * (ket00 + ket11) # 4x1 EPR pair

print(f"|Phi+> = {epr_pair.T.round(3)}") # take transpose and round to 3 decimal places to convert to 1x4 column vector
print(f"Norm check: {np.linalg.norm(epr_pair):.8f}")



|Phi+> = [[0.707+0.j 0.   +0.j 0.   +0.j 0.707+0.j]]
Norm check: 1.00000000


## 3 qubit initial state |𝚿⟩

So we now prepared an initial randomized qubit state for Alice, and an EPR pair from the shared two qubits between Alice and Bob. We take the tensor product of these two states to get the full initial state.

In [ ]:
initial_state = np.kron(alice_psi, epr_pair)

print(f"|Psi> (8x1) =\n{initial_state.round(4)}")
print(f"  Norm check: {np.linalg.norm(initial_state):.8f}")

|Psi> (8x1) =
[[0.6269+0.j]
 [0.    +0.j]
 [0.    +0.j]
 [0.6269+0.j]
 [0.3271+0.j]
 [0.    +0.j]
 [0.    +0.j]
 [0.3271+0.j]]
  Norm check: 1.00000000


# Alice's measurement basis states
From the research meeting notes, the four Bell states used as projectors on Alice's 2-qubit subspace (qubits 0 and 1) are:

* **$|I\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$** $\rightarrow$ $|\Phi^+\rangle$, Identity correction at Bob
* **$|X\rangle = \frac{1}{\sqrt{2}}(|01\rangle + |10\rangle)$** $\rightarrow$ $|\Psi^+\rangle$, $X$ correction at Bob
* **$|Y\rangle = \frac{1}{\sqrt{2}}(|01\rangle - |10\rangle)$** $\rightarrow$ $|\Psi^-\rangle$, $Y$ correction at Bob
* **$|Z\rangle = \frac{1}{\sqrt{2}}(|00\rangle - |11\rangle)$** $\rightarrow$ $|\Phi^-\rangle$, $Z$ correction at Bob

These four states form an orthonormal basis for Alice's $\mathbb{C}^4$ (two-qubit space).

In [ ]:
bell_I = 1/np.sqrt(2) * (ket00 + ket11)
bell_X = 1/np.sqrt(2) * (ket01 + ket10)
bell_Y = 1/np.sqrt(2) * (ket01 - ket10)
bell_Z = 1/np.sqrt(2) * (ket00 - ket11)

bell_states = {'I': bell_I, 'Z': bell_Z, 'X': bell_X, 'Y': bell_Y}

for basis_state, state in bell_states.items():
  print(f"{basis_state}: {state.T.round(3)}")

# proving that it indeed forms an orthonormal basis by computing all 16 inner products
keys = list(bell_states.keys())
gram_schmidt = np.zeros((4,4), dtype=complex)
for i, ki in enumerate(keys):
    for j, kj in enumerate(keys):
        gram_schmidt[i,j] = (bell_states[ki].conj().T @ bell_states[kj])[0,0] # takes conjugate transpose of bra and product wth the ket
print("  Gram-Schmidt matrix (should be identity):")
print(f"{gram_schmidt.real.round(4)}")

I: [[0.707+0.j 0.   +0.j 0.   +0.j 0.707+0.j]]
Z: [[ 0.707+0.j  0.   +0.j  0.   +0.j -0.707+0.j]]
X: [[0.   +0.j 0.707+0.j 0.707+0.j 0.   +0.j]]
Y: [[ 0.   +0.j  0.707+0.j -0.707+0.j  0.   +0.j]]
  Gram-Schmidt matrix (should be identity):
[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]


# Bell measurement using projectors
Applying $P^{\text{label}}$ to $|\Psi\rangle$ gives an unnormalized post-measurement state.
Probability of outcome = $\| P^{\text{label}} |\Psi\rangle \|^2$.

In [ ]:
I2 = np.eye(2, dtype=complex)
measurement_results = {}

for label, state in bell_states.items():
    proj_alice = state @ state.conj().T      # |bell><bell|, shape (4,4)
    proj_full  = np.kron(proj_alice, I2)            # shape (8,8)
    projected  = proj_full @ initial_state            # shape (8,1), unnormalized
    prob       = float(np.real(np.linalg.norm(projected)**2))
    measurement_results[label] = {'prob': prob, 'projected': projected}
    print(f"  P(outcome |{label}>) = {prob:.6f}")

print(f"\n  Total probability: {sum(r['prob'] for r in measurement_results.values()):.8f}  (must be 1.0)")

  P(outcome |I>) = 0.250000
  P(outcome |Z>) = 0.250000
  P(outcome |X>) = 0.250000
  P(outcome |Y>) = 0.250000

  Total probability: 1.00000000  (must be 1.0)


# Bob's measurement extraction

In [ ]:
# After Alice measures outcome |label>, the global state collapses (and normalizes) to:
#   |v> = P^label |Psi> / sqrt(prob)      (8x1 normalized vector)
#
# Since Alice's part is definitively in |label>, we can write:
#   |v> = |label>_{01} ⊗ |phi_label>_B
#
# To extract Bob's 2-component state |phi_label>_B, we use the partial inner product:
#   |phi_label>_B = (<label|_{01} ⊗ I_B) |v>
#
# Implemented as a (2x8) matrix:
#   Extract_B = bell_ket^dag ⊗ I_2
# then:
#   bob_state = Extract_B @ |v>   (shape 2x1)

pauli_I = np.eye(2, dtype=complex)
pauli_X = np.array([[0, 1], [1, 0]], dtype=complex)
pauli_Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
pauli_Z = np.array([[1, 0], [0, -1]], dtype=complex)

# Correction gate Bob applies
#   Outcome I -> Bob has I|psi> = |psi>  -> apply I (no-op)
#   Outcome X -> Bob has X|psi>          -> apply X to invert
#   Outcome Y -> Bob has Y|psi>          -> apply Y (Y^2 = -I ~ I up to phase)
#   Outcome Z -> Bob has Z|psi>          -> apply Z to invert

# maps each bell measurement outcome to the correct standard Pauli matrices
# depending on which state Alice measured, Bob ends up with the Pauli rotated version of the same outcome
# Bob re-applies the same Pauli gate to undo the action and get exactly Alice's state
correction = {'I': pauli_I, 'X': pauli_X, 'Y': pauli_Y, 'Z': pauli_Z}

bob_states = {}
fidelities = {}

for label, res in measurement_results.items():
    prob = res['prob']

    # normalize the projected state
    v_norm = res['projected'] / np.sqrt(prob)         # shape (8,1)

    # Extraction operator: (<label| ⊗ I_2), shape (2,8)
    extract = np.kron(bell_states[label].conj().T, I2)   # (1,4) kron (2,2) -> (2,8)

    bob_raw = extract @ v_norm         # shape (2,1)
    bob_norm_val = np.linalg.norm(bob_raw)

    # Apply Pauli correction to recover |psi>
    bob_corrected = correction[label] @ bob_raw        # shape (2,1)

    bob_states[label] = bob_corrected

    # Fidelity = |<psi|phi_corrected>|^2
    overlap  = (alice_psi.conj().T @ bob_corrected)[0,0]
    fidelity = float(np.abs(overlap)**2)
    fidelities[label] = fidelity

    print(f"  Outcome |{label}> (prob={prob:.4f}):")
    print(f"    Bob's raw |phi_{label}>    = {bob_raw.T.round(6)}")
    print(f"    |phi| norm (sanity)    = {bob_norm_val:.8f}  (must be 1.0)")
    print(f"    After correction gate  = {bob_corrected.T.round(6)}")
    print(f"    Alice's original |psi> = {alice_psi.T.round(6)}")
    print(f"    Fidelity F             = {fidelity:.8f}  (must be 1.0)")

  Outcome |I> (prob=0.2500):
    Bob's raw |phi_I>    = [[0.886597+0.j 0.462543+0.j]]
    |phi| norm (sanity)    = 1.00000000  (must be 1.0)
    After correction gate  = [[0.886597+0.j 0.462543+0.j]]
    Alice's original |psi> = [[0.886597 0.462543]]
    Fidelity F             = 1.00000000  (must be 1.0)
  Outcome |Z> (prob=0.2500):
    Bob's raw |phi_Z>    = [[ 0.886597+0.j -0.462543+0.j]]
    |phi| norm (sanity)    = 1.00000000  (must be 1.0)
    After correction gate  = [[0.886597+0.j 0.462543+0.j]]
    Alice's original |psi> = [[0.886597 0.462543]]
    Fidelity F             = 1.00000000  (must be 1.0)
  Outcome |X> (prob=0.2500):
    Bob's raw |phi_X>    = [[0.462543+0.j 0.886597+0.j]]
    |phi| norm (sanity)    = 1.00000000  (must be 1.0)
    After correction gate  = [[0.886597+0.j 0.462543+0.j]]
    Alice's original |psi> = [[0.886597 0.462543]]
    Fidelity F             = 1.00000000  (must be 1.0)
  Outcome |Y> (prob=0.2500):
    Bob's raw |phi_Y>    = [[-0.462543+0.j  0.88659